# 04. 입찰메이트 RFP RAG — Phase 3 청킹·검색 기준선

유영관 · 2026-11-11

Phase 2에서 파싱한 **100개 파일의 원문·버전·위치**를 검색 가능한 청크로 연결합니다.
같은 원본의 사본은 한 번만 색인하고, 검색 결과에는 원문 범위와 표·시각 검수 상태를 함께 돌려줍니다.

| 이번 단계에서 실행하는 것 | 확인 방법 |
| --- | --- |
| 원문을 보존하는 청킹 | 문서/페이지/섹션 경계, 원문 문자 범위, UTF-8 byte 예산 검사 |
| 검색 기준선 4종 | BM25, 문자 TF-IDF, 가중 RRF, MMR 결과 비교 |
| 선택 문서와 후속 질문 | 사용자 선택 ID와 사용자 질문 이력으로만 검색 |
| 전체 문서 요약 준비 | top-k 대신 해당 문서의 모든 청크를 반환 |
| 검색 평가 | dev에서 정답 근거의 전체 문자 범위를 되찾는지 측정 |

기본 실행에는 OpenAI API 호출·모델 다운로드·GitHub/Drive 변경이 없습니다.
문자 TF-IDF는 **어휘 기반 벡터 검색**이며 의미 임베딩 모델이 아닙니다.
이 단계의 `retrieved`는 근거 후보를 찾았다는 뜻입니다. 생성 답변·요약의 정확도나 5개 업무 기능의 완성도를 뜻하지 않습니다.


## 1. 실행 위치와 선택 옵션

ZIP을 풀고 `notebooks/`에서 엽니다. 다른 위치에서는 `PROJECT_ROOT`를 직접 지정하세요.
동봉된 인덱스·보고서를 읽는 것이 기본값입니다. 의존성이 없으면 먼저 터미널에서
`python -m pip install -r requirements-local.txt`를 실행하세요.
노트북은 설치 명령을 자동 실행하지 않습니다.

`BUILD_INDEX=True`는 현재 설정으로 로컬 청크·인덱스를 다시 구축합니다.
`RUN_DEV_EVALUATION=True`는 dev 검색 실험만 다시 수행합니다.
`ENABLE_OPENAI=True`는 마지막 선택 셀의 외부 임베딩 전송을 켭니다. 처음에는 모두 `False`로 두세요.


In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import importlib.metadata
import json
import os
import sys

PROJECT_ROOT = None  # 예: Path(r'C:\workspace\rfp_phase3_local_20261111')
BUILD_INDEX = False
RUN_DEV_EVALUATION = False
ENABLE_OPENAI = False
SHOW_HOLDOUT_RESULTS = False  # 동결된 평가 보고서 열람만. holdout 질문은 읽지 않음.

if PROJECT_ROOT is None:
    candidates = [Path.cwd(), Path.cwd().parent,
                  Path.cwd() / 'rfp_phase3_local_20261111',
                  Path.cwd() / 'output/rfp_phase3_local_20261111']
    ROOT = next((path.resolve() for path in candidates
                 if (path / 'data/source_documents.jsonl').is_file()
                 and (path / 'local_rag/io.py').is_file()), None)
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if ROOT is None or not (ROOT / 'data/source_documents.jsonl').is_file():
    raise FileNotFoundError('압축을 푼 패키지 폴더를 PROJECT_ROOT에 지정하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

required_dependencies = ['numpy', 'scipy', 'scikit-learn']
dependency_versions = {}
missing_dependencies = []
for name in required_dependencies:
    try:
        dependency_versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        missing_dependencies.append(name)
if missing_dependencies:
    raise ImportError('로컬 의존성이 없습니다: ' + ', '.join(missing_dependencies)
                      + '. 동봉 README의 로컬 설치 안내를 먼저 실행하세요.')

from local_rag.io import load_documents, load_chunks, read_jsonl, sha256_file, verify_saved_run
from local_rag.retrieval import LocalSearchIndex
from local_rag.session import SearchSession

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def show_json(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))

print('작업 폴더:', ROOT.name)
print('의존성:', dependency_versions)
print('로컬 재구축:', BUILD_INDEX, '/ dev 재평가:', RUN_DEV_EVALUATION)
print('OpenAI 실행:', ENABLE_OPENAI, '/ API 키 설정 여부:', bool(os.environ.get('OPENAI_API_KEY')))
print('기본 경로: 로컬 실행 / 자동 설치·원격 파일 변경 없음')


작업 폴더: rfp_phase3_local_20261111
의존성: {'numpy': '2.3.5', 'scipy': '1.17.0', 'scikit-learn': '1.8.0'}
로컬 재구축: False / dev 재평가: False
OpenAI 실행: False / API 키 설정 여부: False
기본 경로: 로컬 실행 / 자동 설치·원격 파일 변경 없음


## 2. 저장 결과와 현재 입력의 버전 맞추기

원문 처리 결과, 문서 목록, 청크, 설정, 코드의 SHA-256을 저장된 실행 기록과 비교합니다.
이 파일 중 하나라도 달라졌으면 이전 인덱스·평가를 그대로 사용하지 않습니다.
재구축은 dev에서 검토하며, holdout 질문을 열어 설정을 고르지 않습니다.

문서 목록의 `sha256`은 Phase 2 원본 파일의 해시입니다. 이번 패키지는 이미 파싱한 원문과 이 해시를 이어받습니다.
PDF/HWP/HWPX 바이너리를 다시 파싱하는 작업은 Phase 2의 로컬 흐름에서 수행하세요.


In [2]:
config = read_json(ROOT / 'configs/default.json')
if BUILD_INDEX:
    from run_phase3 import build_pipeline
    build_pipeline(ROOT, force=True)

required_outputs = ['data/chunks.jsonl', 'reports/run_manifest.json',
                    'reports/phase3_summary.json', 'reports/chunk_integrity.json',
                    'index/index_meta.json']
missing_outputs = [path for path in required_outputs if not (ROOT / path).is_file()]
if missing_outputs:
    raise FileNotFoundError('동봉 결과 누락: ' + ', '.join(missing_outputs)
                            + '. 패키지를 복구하거나 BUILD_INDEX=True로 로컬 재구축하세요.')
recorded_run = verify_saved_run(ROOT)
if not recorded_run.get('files_sha256') or not recorded_run.get('code_sha256'):
    raise ValueError('입력·코드의 버전 기록이 없습니다. 로컬 파이프라인을 다시 구축하세요.')

documents = load_documents(ROOT)
chunks = load_chunks(ROOT)
manifest = read_json(ROOT / 'data/documents_manifest.json')
source_by_id = {document['doc_id']: document for document in documents}
manifest_by_id = {entry['doc_id']: entry for entry in manifest}
if set(source_by_id) != set(manifest_by_id):
    raise ValueError('문서 목록과 파싱 결과의 문서 ID가 다릅니다.')
for doc_id, document in source_by_id.items():
    if document['sha256'] != manifest_by_id[doc_id]['sha256']:
        raise ValueError('문서 목록과 원문 버전이 다릅니다: ' + doc_id)
summary = read_json(ROOT / 'reports/phase3_summary.json')
canonical_ids = {document.get('canonical_doc_id', document['doc_id']) for document in documents}
print('원본 파일:', len(documents), '/ 고유 원문:', len(canonical_ids), '/ 청크:', len(chunks))
print('입력·설정·코드 해시 검사: 통과')
print('청킹 설정:')
show_json(summary.get('chunk_config', config.get('chunking', {})))
print('길이 단위: UTF-8 bytes. 모델 token 수로 해석하지 않습니다.')


원본 파일: 100 / 고유 원문: 99 / 청크: 18266
입력·설정·코드 해시 검사: 통과
청킹 설정:
{
  "chunk_size_bytes": 1800,
  "overlap_bytes": 240
}
길이 단위: UTF-8 bytes. 모델 token 수로 해석하지 않습니다.


## 3. 청킹 정책과 원문 보존 결과

기본 청크 본문 예산은 UTF-8 byte 단위입니다. 실제 값은 위 설정과 보고서에서 확인합니다.
같은 PDF 물리 페이지 또는 HWP/HWPX 섹션 안에서 문단을 묶고, 표는 가능한 한 행을 유지합니다.
큰 블록은 원문 범위를 보존하며 겹쳐 나눕니다. 메타데이터 접두어는 검색을 돕지만 원문 인용으로 쓰지 않습니다.

HWPX의 `reference_blocks`는 과거 인용과의 대조용이므로 별도 본문처럼 다시 색인하지 않습니다.
표 본문도 원문 문단에서 한 번만 청킹합니다. PDF 표 후보의 의미·열 관계는 자동으로 검증됐다고 간주하지 않습니다.
시각 검수된 전사는 `verified_transcription`으로 구분하고, 미검수 OCR은 검색 대상에서 제외합니다.


In [3]:
chunk_counts = Counter(chunk['kind'] for chunk in chunks)
source_method_counts = Counter(chunk.get('source_method') for chunk in chunks)
max_chunk_bytes = max(chunk['text_bytes'] for chunk in chunks)
alias_documents = [document for document in documents
                   if document.get('canonical_doc_id', document['doc_id']) != document['doc_id']]
print('청크 종류:', dict(chunk_counts))
print('원문 처리 방식:', dict(source_method_counts))
print('최대 본문 길이:', max_chunk_bytes, 'UTF-8 bytes')
print('사본으로 연결한 문서:', len(alias_documents))
chunk_audit = summary.get('chunk_audit', {})
chunk_integrity = read_json(ROOT / 'reports/chunk_integrity.json')
audit_keys = ['source_spans_valid', 'all_eligible_source_characters_covered',
              'chunks', 'covered_source_characters', 'uncovered_source_characters']
show_json({key: chunk_audit[key] for key in audit_keys if key in chunk_audit})
print('검색기와 독립적인 원문 보존 검사:')
show_json({key: chunk_integrity[key] for key in
           ['valid', 'source_character_coverage', 'uncovered_non_whitespace_characters']
           if key in chunk_integrity})
print('평가 인용과 청크의 연결:')
alignment_summary = summary.get('evidence_alignment', {})
show_json({key: value for key, value in alignment_summary.items()
           if key not in {'records', 'document_aliases', 'chunk_source_issues'}})
if chunk_integrity.get('valid') is not True:
    raise ValueError('원문 보존 검사 실패. reports의 상세 결과를 확인하세요.')
assert all(chunk['doc_id'] in canonical_ids for chunk in chunks)
assert all(chunk.get('length_unit') == 'utf8_bytes' for chunk in chunks)
assert all(chunk['text_bytes'] == len(chunk['text'].encode('utf-8')) for chunk in chunks)
assert all(span.get('collection') in {'blocks', 'supplemental_blocks'}
           for chunk in chunks for span in chunk['source_spans'])


청크 종류: {'prose': 9725, 'table': 8538, 'verified_transcription': 3}
원문 처리 방식: {'raw_parser': 18263, 'reviewed_visual_transcription': 3}
최대 본문 길이: 1800 UTF-8 bytes
사본으로 연결한 문서: 1
{
  "source_spans_valid": true,
  "all_eligible_source_characters_covered": true,
  "chunks": 18266,
  "covered_source_characters": 6176884,
  "uncovered_source_characters": 0
}
검색기와 독립적인 원문 보존 검사:
{
  "valid": true,
  "source_character_coverage": 1.0,
  "uncovered_non_whitespace_characters": 0
}
평가 인용과 청크의 연결:
{
  "total_links": 684,
  "covered_links": 684,
  "uncovered_links": 0,
  "source_ambiguous_links": 1,
  "chunk_source_issue_count": 0,
  "valid": true
}


## 4. 실제 청크에서 원문 위치로 돌아가기

아래는 표·시각 검수 전사·일반 본문을 하나씩 표시합니다.
청크의 `source_spans`에 저장한 문자 범위가 원본 블록의 같은 문자열인지 직접 비교합니다.
`char_start/end`와 `chunk_start/end`는 Python Unicode 문자 오프셋입니다. byte 예산과 구분합니다.
이 검사와 청킹 보고서는 이미 파싱된 텍스트의 보존을 검증하며, PDF 그림 안의 모든 내용이 추출됐다는 뜻은 아닙니다.


In [4]:
block_lookup = {}

def original_block(chunk, span):
    doc_id = chunk['doc_id']
    collection = span['collection']
    key = (doc_id, collection)
    if key not in block_lookup:
        block_lookup[key] = {block['block_id']: block
                             for block in source_by_id[doc_id].get(collection, [])}
    return block_lookup[key][span['block_id']]

examples = []
for kind in ['table', 'verified_transcription', 'prose']:
    found = next((chunk for chunk in chunks if chunk['kind'] == kind), None)
    if found:
        examples.append(found)
for chunk in examples:
    for span in chunk['source_spans']:
        block = original_block(chunk, span)
        assert block['text'][span['char_start']:span['char_end']] == \
               chunk['text'][span['chunk_start']:span['chunk_end']]
        assert block['locator'] == span['locator']
    print('종류:', chunk['kind'], '/ 파일:', chunk['file_name'])
    print('청크:', chunk['chunk_id'], '/ 위치:', chunk['locator'])
    print('처리 방식:', chunk.get('source_method'), '/ 품질 표시:', chunk.get('quality_flags', []))
    print('표 상태:', chunk.get('structure_status'), '/ 본문 bytes:', chunk['text_bytes'])
    print('첫 원문 범위:', {key: chunk['source_spans'][0].get(key)
                           for key in ['collection', 'block_id', 'locator', 'char_start', 'char_end']})
    print(chunk['text'][:600])
    print()
print('표시한 청크의 전체 source span 원문 일치: 통과')


종류: table / 파일: 서울특별시_2026_2027 서울패션위크 트레이드쇼 운영 용역_제안요청서.hwpx
청크: rfp:03a15712e8e9bd5e:d97b0763ce8988801b19946a / 위치: Contents/section0.xml / paragraph 6 / segment 1 … Contents/section0.xml / paragraph 9 / segment 1
처리 방식: raw_parser / 품질 표시: ['FIRST_ROW_CONTEXT_IS_NOT_A_VERIFIED_COLUMN_HEADER']
표 상태: explicit / 본문 bytes: 94
첫 원문 범위: {'collection': 'blocks', 'block_id': 'hwpx:s0:p000006:seg01', 'locator': 'Contents/section0.xml / paragraph 6 / segment 1', 'char_start': 0, 'char_end': 5}
과 업 명
2027 서울패션위크 트레이드쇼 운영 용역
주관기관
서울특별시

종류: verified_transcription / 파일: 한국예탁결제원_2024_차세대시스템(1단계) 구축 사업_제안요청서.pdf
청크: rfp:10574559caefec03:740bb4f8e9b6360f9df6e92a / 위치: PDF p.3 / 시각 검수 전사 KSD_P3_NAME
처리 방식: reviewed_visual_transcription / 품질 표시: ['PDF_TABLE_CANDIDATES_NOT_VISUALLY_VERIFIED', 'document:PDF_TABLES_HEURISTIC', 'document:VISUAL_TRANSCRIPTION_APPLIED']
표 상태: heuristic / 본문 bytes: 53
첫 원문 범위: {'collection': 'supplemental_blocks', 'block_id': 'transcription_KSD_P3_NAME', 'locator': 'PDF p.3

## 5. 영속 인덱스와 캐시 확인

저장된 인덱스는 본문·설정·검색 코드·의존성 버전과 내부 파일 해시를 검사합니다.
이전에 저장한 행 수가 같더라도 내용이 바뀌었으면 재사용하지 않습니다.
환경의 numpy/scipy/scikit-learn 버전이 달라 저장 인덱스가 거부되면 현재 환경에서 다시 구축하세요.

검색 전처리는 원문과 별도로 적용됩니다. 띄어 쓴 한글 표제나 예산 관련 표현을 검색용으로 보조하더라도
인용할 원문 문자열과 숫자는 수정하지 않습니다.


In [5]:
import sys
import json
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("실행 중인 Python:", sys.executable)
print("Python 버전:", sys.version.split()[0])

root = globals().get("ROOT")
print("프로젝트 경로:", root)

saved = {}
if root is not None:
    path = Path(root) / "index" / "index_meta.json"
    if path.is_file():
        saved = json.loads(path.read_text(encoding="utf-8"))["dependencies"]

for name in ["numpy", "scipy", "scikit-learn"]:
    try:
        current = version(name)
    except PackageNotFoundError:
        current = "설치되지 않음"

    expected = saved.get(name, "확인 불가")
    print(f"{name}: 저장된 버전={expected}, 현재 버전={current}, 일치={expected == current}")

실행 중인 Python: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase3_local_20261111/.venv/bin/python
Python 버전: 3.11.7
프로젝트 경로: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase3_local_20261111
numpy: 저장된 버전=2.3.5, 현재 버전=2.3.5, 일치=True
scipy: 저장된 버전=1.17.0, 현재 버전=1.17.0, 일치=True
scikit-learn: 저장된 버전=1.8.0, 현재 버전=1.8.0, 일치=True


In [6]:
index = LocalSearchIndex(ROOT / 'index')
index_metadata = index.get_metadata()
chunk_digest = hashlib.sha256()
for chunk in chunks:
    encoded = json.dumps(chunk, ensure_ascii=False, sort_keys=True,
                         separators=(',', ':'), allow_nan=False).encode('utf-8')
    chunk_digest.update(len(encoded).to_bytes(8, 'big'))
    chunk_digest.update(encoded)
assert chunk_digest.hexdigest() == index_metadata['chunks_fingerprint'], \
       '현재 청크와 저장 인덱스가 다릅니다. BUILD_INDEX=True로 재구축하세요.'
assert index_metadata['chunk_count'] == len(chunks)
assert index_metadata['canonical_document_count'] == len(canonical_ids)
catalog = index.get_catalog()
assert {row['doc_id'] for row in catalog} == canonical_ids
for row in catalog:
    assert row['sha256'] == source_by_id[row['doc_id']]['sha256']
print('인덱스 문서:', len(catalog), '/ 청크:', index_metadata['chunk_count'])
print('문자 TF-IDF 특성 수:', index_metadata['feature_count'])
print('인덱스 fingerprint:', index_metadata['index_fingerprint'])
print('원문 버전·청크 내용·인덱스 내부 파일·코드·의존성 검사: 통과')
print('점수 의미:')
show_json(index_metadata['score_semantics'])
print('검색 점수는 보정된 확률이나 답변 신뢰도가 아닙니다.')


인덱스 문서: 99 / 청크: 18266
문자 TF-IDF 특성 수: 100000
인덱스 fingerprint: b784f5cd9f85f178f72c1faf99ebdfc89d97684e35c81b3dfc1563d0a1907fea
원문 버전·청크 내용·인덱스 내부 파일·코드·의존성 검사: 통과
점수 의미:
{
  "bm25": "negative SQLite BM25 rank, larger is better",
  "char_tfidf": "character lexical TF-IDF cosine similarity",
  "rrf": "weighted reciprocal rank fusion",
  "mmr": "relevance/diversity objective using lexical TF-IDF redundancy"
}
검색 점수는 보정된 확률이나 답변 신뢰도가 아닙니다.


## 6. 같은 실제 질문으로 검색 방법 비교

dev 질문의 원래 `question`과 실제 사용자 선택인 `context.selected_doc_ids`만 검색기에 전달합니다.
`target_doc_ids`, 필수 사실, 정답 금액·날짜를 검색어나 문서 필터에 넣지 않습니다.
후보 수 `fetch_k`와 최종 결과 수 `top_k`를 분리합니다.

| 방법 | 점수의 뜻 |
| --- | --- |
| BM25 | 검색용 단어/문자 n-gram의 어휘 순위 점수 |
| 문자 TF-IDF | 문자 n-gram 어휘 벡터의 cosine 유사도 |
| RRF | BM25와 TF-IDF의 순위를 가중 역순위 합으로 결합 |
| MMR | RRF 후보에서 관련도와 TF-IDF 중복도를 함께 고려 |

방법별 점수 단위가 다르므로 점수 크기를 방법 사이에서 비교하지 않습니다.
아래 출력은 생성 답변이 아니라 원문 근거 후보입니다.


In [7]:
dev_cases = read_jsonl(ROOT / 'eval/questions.dev.jsonl')
assert not any(case.get('split') == 'holdout' for case in dev_cases)
fact_case = next(case for case in dev_cases
                 if case.get('feature') == 'fact'
                 and case.get('context', {}).get('selected_doc_ids'))
query = fact_case['question']
selected_ids = list(fact_case['context']['selected_doc_ids'])
TOP_K = config['search']['top_k']
FETCH_K = config['search']['fetch_k']
MMR_LAMBDA = config['search']['mmr_lambda']
print('dev 사례:', fact_case['case_id'])
print('질문:', query)
print('사용자 선택 문서:', selected_ids)

def show_hit(hit, preview_chars=360):
    print('순위:', hit.get('rank'), '/ 방법:', hit.get('method'),
          '/ 점수:', round(float(hit.get('score', 0)), 7),
          '/ 점수 종류:', hit.get('score_kind'))
    print('파일:', hit['file_name'])
    print('문서:', hit['doc_id'], '/ 청크:', hit['chunk_id'])
    print('원본 SHA-256:', hit['sha256'])
    print('위치:', hit['locator'])
    print('처리 방식:', hit.get('source_method'), '/ 품질 표시:', hit.get('quality_flags', []))
    print('원문 범위:', [{key: span.get(key)
                         for key in ['block_id', 'locator', 'char_start', 'char_end']}
                        for span in hit['source_spans'][:2]])
    print(hit['text'][:preview_chars])

method_hits = {}
allowed_canonical = set(index.resolve_doc_ids(selected_ids))
for method in ['bm25', 'char_tfidf', 'rrf', 'mmr']:
    hits = index.search(query, top_k=TOP_K, allowed_doc_ids=selected_ids,
                        method=method, fetch_k=FETCH_K, mmr_lambda=MMR_LAMBDA)
    method_hits[method] = hits
    assert all(hit['doc_id'] in allowed_canonical for hit in hits)
    assert all(hit.get('score_is_confidence') is False for hit in hits)
    print('방법:', method, '/ 후보 상한:', FETCH_K, '/ 반환:', len(hits))
    if hits:
        show_hit(hits[0])
    print()
print('빈 결과는 문서에 해당 사실이 없다는 확정이 아닙니다.')


dev 사례: F001
질문: 2027년 글로벌 패션브랜드 육성 용역의 용역비는 얼마이며 부가세가 포함되나요?
사용자 선택 문서: ['1Uy4SDf7e_NFKK9CbjL9uVO4ta_MQdeTt']
방법: bm25 / 후보 상한: 40 / 반환: 5
순위: 1 / 방법: bm25 / 점수: 134.9589331 / 점수 종류: bm25_rank_score_not_confidence
파일: 서울특별시_2026_2027년 글로벌 패션브랜드 육성 용역_제안요청서.hwpx
문서: 1Uy4SDf7e_NFKK9CbjL9uVO4ta_MQdeTt / 청크: rfp:081719865dd99a38:2d6f15607efe3625752ac288
원본 SHA-256: 081719865dd99a38ba822b39869efcf75b37add88f8d07741c8f8bf141060484
위치: Contents/section0.xml / paragraph 64 / segment 1 … Contents/section0.xml / paragraph 77 / segment 1
처리 방식: raw_parser / 품질 표시: []
원문 범위: [{'block_id': 'hwpx:s0:p000064:seg01', 'locator': 'Contents/section0.xml / paragraph 64 / segment 1', 'char_start': 0, 'char_end': 8}, {'block_id': 'hwpx:s0:p000065:seg01', 'locator': 'Contents/section0.xml / paragraph 65 / segment 1', 'char_start': 0, 'char_end': 31}]
가. 과업 개요
ㅇ 과 업 명 : 2027년 글로벌 패션브랜드 육성 용역
ㅇ 과업기간 : 2026. 12. ~ 2027. 11. 중 프로그램 진행 기간 10개월
- 상 반 기 : ’26. 11. 30. ~ ’27. 4. 30.  (5개월)
- 하 반 기 : ’27. 7. 1. ~ ’2

## 7. 복수 문서 범위와 사본 ID

문서 비교에서도 사용자가 선택한 두 문서만 검색합니다.
통합 top-k에는 한쪽 문서가 덜 나타날 수 있으므로, 이후 비교 답변 단계에서는 문서별 근거 확보 여부를 따로 확인해야 합니다.
사본 ID는 같은 원문의 canonical ID로 해석하지만, 알 수 없는 ID를 전체 문서 검색으로 바꾸지 않습니다.
작성연도와 사업연도가 다르다는 이유만으로 문서를 제외하지 않습니다.


In [8]:
comparison_case = next(case for case in dev_cases
                       if case.get('feature') == 'comparison'
                       and len(case.get('context', {}).get('selected_doc_ids', [])) >= 2)
comparison_ids = list(comparison_case['context']['selected_doc_ids'])
comparison_hits = index.search(comparison_case['question'], top_k=6,
                               allowed_doc_ids=comparison_ids, method='rrf', fetch_k=FETCH_K,
                               mmr_lambda=MMR_LAMBDA)
comparison_scope = set(index.resolve_doc_ids(comparison_ids))
assert all(hit['doc_id'] in comparison_scope for hit in comparison_hits)
print('dev 사례:', comparison_case['case_id'])
print('질문:', comparison_case['question'])
print('사용자 선택 문서 수:', len(comparison_scope))
print('검색 결과의 문서별 청크 수:', dict(Counter(hit['doc_id'] for hit in comparison_hits)))
for hit in comparison_hits[:2]:
    show_hit(hit, preview_chars=240)
    print()
for document in alias_documents:
    resolved = index.resolve_doc_ids([document['doc_id']])
    assert resolved == [document['canonical_doc_id']]
    print('사본 ID 연결:', document['doc_id'], '→', resolved[0])


dev 사례: C001
질문: 선택한 다음 두 RFP를 비교해 줘: 「한국주택금융공사: HF 인사이트 시스템 개발」 / 「한국주택금융공사: 부동산 등기정보 시스템 구축」. 비교 항목은 예산(부가세 및 개발·HW 내역이 있으면 포함)과 사업기간(안정화·조정 조건 포함)야. 문서별 열을 두는 표로 정리하고 각 데이터 칸에 근거를 붙여 줘. 기간이나 예산의 기준이 다르면 그대로 설명하고 우열 추천은 하지 마.
사용자 선택 문서 수: 2
검색 결과의 문서별 청크 수: {'1G2at8Un7y8JIn5awuVxqV7OOKIj7zT4B': 2, '1MxTCLhVvegbCCvqonlf95yO6pr4fYgaJ': 4}
순위: 1 / 방법: rrf / 점수: 0.0308349 / 점수 종류: weighted_reciprocal_rank_fusion_not_confidence
파일: 한국주택금융공사_2023_HF 인사이트 시스템 개발_제안요청서.hwp
문서: 1G2at8Un7y8JIn5awuVxqV7OOKIj7zT4B / 청크: rfp:8c293ac2bb077438:b5c08edd8c31f7f8d0bef61a
원본 SHA-256: 8c293ac2bb077438c0aaa974746d203a9196ec3eaa0d2c7971e5f6d3e4a28e28
위치: BodyText/Section0 / paragraph 10 … BodyText/Section0 / paragraph 13
처리 방식: raw_parser / 품질 표시: ['FIRST_ROW_CONTEXT_IS_NOT_A_VERIFIED_COLUMN_HEADER', 'document:HWP_LAYOUT_NOT_RENDERED', 'document:NON_TEXT_OBJECTS_PRESENT']
원문 범위: [{'block_id': 'BodyText/Section0:paragraph:10:record:1603', 'locator': 'BodyText/Section0 / paragraph 10', 'char_start': 0, 'cha

## 8. 후속 질문에서 선택 문서 유지

검색 세션은 사용자 선택과 사용자 질문 이력을 보관합니다.
“그건” 같은 후속 질문에는 같은 선택 범위의 직전 **사용자 질문**을 보조 문맥으로 사용합니다.
과거 assistant 답변이나 평가 정답 사실은 이력에 주입하지 않습니다.
문서 선택을 바꾸면 이전 사업의 질문을 새 검색어에 덧붙이지 않습니다.

아래는 dev 대화 사례의 실제 첫 두 사용자 발화만 실행합니다. `retrieved`와 정답의 `answered`를 동일하게 채점하지 않습니다.


In [9]:
followup_case = next(case for case in dev_cases
                     if case.get('feature') == 'followup'
                     and len(case.get('turns', [])) >= 2
                     and case.get('context', {}).get('selected_doc_ids'))
session = SearchSession(index, active_doc_ids=followup_case['context']['selected_doc_ids'])
followup_results = []
for turn in followup_case['turns'][:2]:
    result = session.ask(turn['user'], top_k=3, method='rrf')
    followup_results.append(result)
    print('사용자:', turn['user'])
    print('상태:', result['status'], '/ 선택 문서:', result['active_doc_ids'])
    print('독립 검색 질의:', result['standalone_query'])
    print('반환 청크:', len(result['hits']), '/ 생성 답변:', result['answer_generated'])
    print()
assert session.query_history == [turn['user'] for turn in followup_case['turns'][:2]]
assert all(result['retrieval_only'] and not result['answer_generated'] for result in followup_results)
independent_session = SearchSession(index)
assert independent_session.query_history == []
assert independent_session.active_doc_ids == []

switch_id = next(doc_id for doc_id in comparison_ids if doc_id not in session.active_doc_ids)
session.select_documents([switch_id])
switched = session.ask('그 문서의 수행기간은?', top_k=3)
assert '직전 사용자 질문:' not in switched['standalone_query']
assert set(hit['doc_id'] for hit in switched['hits']).issubset(set(index.resolve_doc_ids([switch_id])))
print('선택 변경 후 질의:', switched['standalone_query'])
print('선택 변경 시 이전 사업의 검색 문맥 초기화: 통과')
print('새 세션의 선택·질문 이력 분리: 통과')


사용자: 한국주택금융공사의 HF 인사이트 시스템 개발 예산이 얼마야? 세금 포함 여부도 알려줘.
상태: retrieved / 선택 문서: ['1G2at8Un7y8JIn5awuVxqV7OOKIj7zT4B']
독립 검색 질의: 한국주택금융공사의 HF 인사이트 시스템 개발 예산이 얼마야? 세금 포함 여부도 알려줘.
현재 선택 사업: HF 인사이트 시스템 개발
반환 청크: 3 / 생성 답변: False

사용자: 그건 몇 개월짜리야? 안정화도 그 안에 들어가?
상태: retrieved / 선택 문서: ['1G2at8Un7y8JIn5awuVxqV7OOKIj7zT4B']
독립 검색 질의: 그건 몇 개월짜리야? 안정화도 그 안에 들어가?
현재 선택 사업: HF 인사이트 시스템 개발
직전 사용자 질문: 한국주택금융공사의 HF 인사이트 시스템 개발 예산이 얼마야? 세금 포함 여부도 알려줘.
반환 청크: 3 / 생성 답변: False

선택 변경 후 질의: 그 문서의 수행기간은?
현재 선택 사업: 부동산 등기정보 시스템 구축
선택 변경 시 이전 사업의 검색 문맥 초기화: 통과
새 세션의 선택·질문 이력 분리: 통과


## 9. 전체 문서 요약은 모든 청크를 가져오는 별도 경로

전체 요약에 일반 검색 top-k만 넣으면 예산·자격·기한·평가기준을 빠뜨릴 수 있습니다.
아래는 실제 사용자 선택 문서의 **모든 인덱싱된 청크**를 순서대로 반환합니다.
이후 생성 단계에서는 섹션별 요약과 전체 항목 검사를 연결할 수 있습니다.

`structural_chunk_coverage=1.0`은 인덱스 안의 청크를 모두 반환했다는 뜻입니다.
원본 그림·표의 의미를 100% 읽었다거나 요약을 생성·검증했다는 뜻은 아닙니다.


In [10]:
summary_case = next(case for case in dev_cases
                    if case.get('feature') == 'summary'
                    and case.get('context', {}).get('selected_doc_ids'))
summary_doc_id = summary_case['context']['selected_doc_ids'][0]
summary_session = SearchSession(index, active_doc_ids=[summary_doc_id])
full_document = summary_session.get_full_document_chunks(summary_doc_id)
assert len(full_document['chunks']) == len(index.get_document_chunks(summary_doc_id))
assert all(chunk['doc_id'] == full_document['doc_id'] for chunk in full_document['chunks'])
assert not full_document['answer_generated']
print('dev 사례:', summary_case['case_id'])
print('전체 문서 경로:')
show_json({key: value for key, value in full_document.items() if key != 'chunks'})
print('처음 위치:', full_document['chunks'][0]['locator'] if full_document['chunks'] else None)
print('마지막 위치:', full_document['chunks'][-1]['locator'] if full_document['chunks'] else None)
print('요약 생성·채점: 아직 실행하지 않음')


dev 사례: S001
전체 문서 경로:
{
  "status": "document_chunks",
  "doc_id": "1wVR2B8RyCsod3YKjxacOb0i-wnUEFdlK",
  "active_doc_ids": [
    "1wVR2B8RyCsod3YKjxacOb0i-wnUEFdlK"
  ],
  "returned_chunks": 22,
  "total_indexed_chunks": 22,
  "structural_chunk_coverage": 1.0,
  "coverage_basis": "all_chunks_returned_by_index_for_selected_document",
  "full_original_semantic_coverage_verified": false,
  "answer_generated": false,
  "language_model_used": false
}
처음 위치: BodyText/Section0 / paragraph 10 … BodyText/Section0 / paragraph 11
마지막 위치: BodyText/Section0 / paragraph 206 … BodyText/Section0 / paragraph 219
요약 생성·채점: 아직 실행하지 않음


## 10. dev 검색 성능과 실패 사례

주 평가는 사실 추출·문서 비교의 정답 근거를 검색했는지 봅니다.
`evidence_span_recall`은 인용문의 비공백 원문 문자를 top-k 청크의 합으로 **전부** 복구한 비율입니다.
인용문 일부만 겹쳤다고 성공으로 세지 않습니다. `MRR_first_source_overlap`은 첫 부분 겹침의 순위여서 더 약한 지표입니다.
지연시간은 이 실행 환경의 로컬 검색 시간이며 OpenAI 응답시간이 아닙니다.

`context`는 실제 사용자 선택 문서로 한정한 평가, `global`은 문서 필터가 없는 평가입니다.
두 조건을 분리해서 비교합니다. 요약·대화·거절을 포함한 5개 기능의 종합 점수로 해석하지 않습니다.
아래는 동봉 dev 결과만 기본 표시합니다. holdout 결과를 본 뒤 설정을 조정하지 마세요.


In [11]:
dev_report_path = ROOT / 'reports/dev_evaluation.json'
dev_report = read_json(dev_report_path) if dev_report_path.is_file() else {}

def report_leaves(value, path=()):
    if isinstance(value, dict):
        if 'metrics_at_k' in value:
            yield path, value
        else:
            for key, child in value.items():
                if key not in {'case_results', 'records'}:
                    yield from report_leaves(child, path + (str(key),))
    elif isinstance(value, list):
        for number, child in enumerate(value):
            yield from report_leaves(child, path + (str(number),))

def show_retrieval_report(report):
    leaves = list(report_leaves(report))
    if not leaves:
        show_json(report)
        return
    for path, result in leaves:
        print('방법:', result.get('method'), '/ 범위:', result.get('scope_protocol'),
              '/ 사례:', result.get('case_count'))
        for k, metrics in result['metrics_at_k'].items():
            print('k =', k, {key: metrics.get(key) for key in
                  ['micro_evidence_span_recall', 'mean_document_recall', 'complete_cases']})
        print('MRR_first_source_overlap:', result.get('mrr_first_source_overlap'))
        print('로컬 지연시간 ms:', result.get('latency_ms'))
        print()

if dev_report:
    show_retrieval_report(dev_report)
else:
    print('현재 설정의 dev 평가 보고서가 없습니다. 아래 선택 셀에서 RUN_DEV_EVALUATION=True로 측정하세요.')
rrf_context_report = next((result for _, result in report_leaves(dev_report)
                           if result.get('method') == 'rrf'
                           and result.get('scope_protocol') == 'context'), None)
if rrf_context_report:
    failed_cases = [row for row in rrf_context_report.get('case_results', [])
                    if not row.get('metrics', {}).get(str(TOP_K), {}).get('all_required_evidence_recovered')]
    print('RRF context에서 top-k 근거가 완전하지 않은 dev 사례:', len(failed_cases))
    for row in failed_cases[:3]:
        print(row['case_id'], row['query'])
        print('검색에 사용한 선택 문서:', row.get('user_selected_doc_ids'))
        print('검색 결과 청크:', row.get('retrieved_chunk_ids', [])[:TOP_K])
        print('top-k 근거 지표:', row.get('metrics', {}).get(str(TOP_K)))
if SHOW_HOLDOUT_RESULTS:
    holdout_path = ROOT / 'reports/holdout_evaluation.json'
    if holdout_path.is_file():
        holdout_report = read_json(holdout_path)
        print('설정 동결 뒤 한 번 평가한 holdout 보고서. 이후 설정 선택에 사용하지 마세요.')
        show_retrieval_report(holdout_report)
    else:
        print('현재 설정의 동결된 holdout 보고서가 없습니다. 이 노트북에서는 holdout을 재평가하지 않습니다.')
else:
    print('holdout 질문·정답 및 상세 평가 결과를 열지 않았습니다.')


방법: bm25 / 범위: context / 사례: 27
k = 5 {'micro_evidence_span_recall': 0.5955056179775281, 'mean_document_recall': 0.9074074074074074, 'complete_cases': 12}
k = 10 {'micro_evidence_span_recall': 0.7191011235955056, 'mean_document_recall': 0.9629629629629629, 'complete_cases': 16}
k = 20 {'micro_evidence_span_recall': 0.8314606741573034, 'mean_document_recall': 0.9629629629629629, 'complete_cases': 21}
MRR_first_source_overlap: 0.6401666234999568
로컬 지연시간 ms: {'p50': 31.115757999941707, 'p95': 60.151427000164404}

방법: char_tfidf / 범위: context / 사례: 27
k = 5 {'micro_evidence_span_recall': 0.24719101123595505, 'mean_document_recall': 0.8888888888888888, 'complete_cases': 8}
k = 10 {'micro_evidence_span_recall': 0.29213483146067415, 'mean_document_recall': 0.8888888888888888, 'complete_cases': 9}
k = 20 {'micro_evidence_span_recall': 0.34831460674157305, 'mean_document_recall': 0.8888888888888888, 'complete_cases': 9}
MRR_first_source_overlap: 0.26825396825396824
로컬 지연시간 ms: {'p50': 2.3890940

### 선택: 현재 설정을 dev에서 다시 측정

이 셀은 `RUN_DEV_EVALUATION=True`일 때만 실행합니다.
정답 근거는 검색이 끝난 뒤 결과를 채점하는 label로만 사용합니다.
원래 질문과 선택 문서 이외의 정답 값·기대 문서 ID는 검색기에 전달하지 않습니다.
동봉된 공식 보고서를 덮어쓰지 않으며, holdout 파일을 읽거나 다시 평가하지 않습니다.


In [12]:
dev_rerun_results = {}
if RUN_DEV_EVALUATION:
    from local_rag.evaluation import audit_chunk_coverage, evaluate_retrieval
    source_evidence = read_json(ROOT / 'data/source_evidence_validation.json')
    dev_alignment = audit_chunk_coverage(documents, chunks, source_evidence)
    if not dev_alignment['valid']:
        raise ValueError('원문 근거와 청크의 연결 검사 실패. 검색 평가 전에 연결을 수정하세요.')
    for method in config['evaluation']['methods']:
        dev_rerun_results[method] = {}
        for scope_mode in config['evaluation']['scope_modes']:
            result = evaluate_retrieval(index, dev_cases, dev_alignment,
                                        top_ks=config['evaluation']['top_ks'], method=method,
                                        scope_mode=scope_mode, search_config=config['search'])
            dev_rerun_results[method][scope_mode] = result
    show_retrieval_report(dev_rerun_results)
else:
    print('dev 재평가 생략: 동봉 reports/dev_evaluation.json이 있으면 해당 결과 사용')


dev 재평가 생략: 동봉 reports/dev_evaluation.json이 있으면 해당 결과 사용


## 11. 선택: OpenAI 의미 임베딩 + 로컬 Chroma

실제 API 성능 비교는 아직 측정하지 않았습니다. 기본 실행의 외부 API 호출은 0회입니다.
외부 임베딩을 실험하려면 별도 환경에 `requirements-openai.txt`의 의존성을 수동 설치하고,
키를 노트북에 쓰지 말고 프로세스 환경변수 `OPENAI_API_KEY`로 설정하세요.

`ENABLE_OPENAI=True`이면 아래 선택 문서 청크와 질의가 OpenAI 임베딩 API로 전송됩니다.
기본은 위 사실 질문에서 사용자가 선택한 공개 문서 1개입니다. 문서를 바꾸면 공개 전송 범위와 원본 버전을 먼저 확인하세요.
문서 목록에서 공개 범위가 기록된 SHA-256만 허용합니다.

`text-embedding-3-small` / 1,536차원으로 문서와 질의를 동일하게 임베딩합니다.
로컬 Chroma를 영속 저장하며, 모델·차원·청크 내용이 다르면 캐시를 재사용하지 않습니다.
예산 값은 보수적인 **UTF-8 byte 기반 token 상한**이며 실제 token 사용량이나 금액이 아닙니다.
입력·요청·세션 상한을 첫 호출 전에 점검하고, 초과하면 멈춥니다. 전체 문서로 확대할 때는 사용량을 검토한 뒤 별도 설정하세요.

참조: [OpenAI Embeddings API](https://developers.openai.com/api/reference/python/resources/embeddings/methods/create)
이 셀은 의미 검색 연결 코드입니다. LLM 답변 생성은 다음 단계에서 연결합니다.


In [13]:
openai_build_report = None
openai_hits = []
OPENAI_DOC_IDS = selected_ids[:1]
OPENAI_TOTAL_TOKEN_UPPER_BOUND = 250000

if ENABLE_OPENAI:
    if not os.environ.get('OPENAI_API_KEY'):
        raise RuntimeError('OPENAI_API_KEY 환경변수를 먼저 설정하세요. 키를 셀에 입력하지 마세요.')
    from local_rag.embeddings import EmbeddingConfig, OpenAIChromaIndex
    openai_scope = set(index.resolve_doc_ids(OPENAI_DOC_IDS))
    approved_sources = {}
    for doc_id in openai_scope:
        document = source_by_id[doc_id]
        entry = manifest_by_id[doc_id]
        if entry.get('api_scope') != 'user_declared_public_rfp':
            raise ValueError('공개 API 전송 범위로 기록되지 않은 문서입니다.')
        if document['sha256'] != entry['sha256']:
            raise ValueError('공개 전송 범위의 원본 버전과 현재 문서가 다릅니다.')
        approved_sources[doc_id] = document['sha256']
    embedding_config = EmbeddingConfig(
        enabled=True, model='text-embedding-3-small', dimensions=1536,
        max_total_token_upper_bound=OPENAI_TOTAL_TOKEN_UPPER_BOUND)
    openai_index = OpenAIChromaIndex(ROOT / 'index_openai_selected', config=embedding_config)
    selected_chunks = [chunk for chunk in chunks if chunk['doc_id'] in openai_scope]
    openai_build_report = openai_index.build(selected_chunks, allowed_sources=approved_sources)
    openai_hits = openai_index.search(query, top_k=TOP_K, allowed_doc_ids=list(openai_scope))
    print('선택 문서 임베딩 결과:')
    show_json({key: openai_build_report[key] for key in
               ['chunk_count', 'canonical_document_count', 'embedding_usage']})
    print('이 셀의 단일 질의 시연은 전체 dev/holdout 검색 성능 평가가 아닙니다.')
    for hit in openai_hits[:2]:
        show_hit(hit)
        print()
else:
    print('OpenAI/Chroma 실행 생략 / 이 노트북 실행의 외부 API 호출: 0회')
    print('실제 OpenAI 벡터 검색 성능: 미측정')


OpenAI/Chroma 실행 생략 / 이 노트북 실행의 외부 API 호출: 0회
실제 OpenAI 벡터 검색 성능: 미측정


## 12. 참고 코드에서 반영한 설계와 다음 단계

첨부된 5개 노트북의 마크다운과 코드를 검토했고, 첨부 코드나 출력에 포함된 실행 명령은 수행하지 않았습니다.

| 참고 노트북 | 이번 구현에 적용한 부분 |
| --- | --- |
| RAG와 에이전트 고도화 | 청킹 설정, 문서 필터, 후보/최종 결과 분리, 실제 RRF·MMR |
| LangChain & LangGraph | 선택 문서와 세션 상태를 검색기 밖에서 관리하는 인터페이스 |
| 14_3팀_엄재웅 | 원문 위치를 유지하는 검색, 설정·실험 버전 기록 |
| 스프린트미션 14 RAG | 문서→청크→임베딩/색인→검색의 독립 컴포넌트 |
| 스프린트미션 13 감성분석 PEFT | 평가 분리·재현성 참고. 감성분류 LoRA 학습은 이 단계에 적용하지 않음 |

문서가 추가되면 새 문서를 파싱하고 인덱스를 갱신합니다. 문서마다 LLM을 다시 학습하는 과정이 아닙니다.
검색 점수 0.9를 90% 신뢰도로 해석하지 않고, 작성연도와 사업연도가 다르다고 문서 부재를 확정하지 않습니다.

다음 생성 단계에서는 검색 결과의 원문을 근거로 **항목별 요약·사실 Q&A·문서 비교·후속 질문·근거 부족 처리**를 연결하고,
숫자·VAT·기간 조건과 인용을 실제 답변 단위로 평가합니다.
LangGraph와 추가 업무 도구는 검증된 검색·생성 함수를 도구 인터페이스로 연결하면 됩니다.


In [14]:
completion_checks = {
    'source_ids_and_versions_match': set(source_by_id) == set(manifest_by_id)
        and all(document['sha256'] == manifest_by_id[doc_id]['sha256']
                for doc_id, document in source_by_id.items()),
    'canonical_content_indexed_once': {chunk['doc_id'] for chunk in chunks} == canonical_ids
        and len(catalog) == len(canonical_ids),
    'chunk_and_index_content_match': chunk_digest.hexdigest() == index_metadata['chunks_fingerprint'],
    'source_character_audit_passed': chunk_integrity.get('valid') is True,
    'actual_dev_query_used': query == fact_case['question'],
    'selected_scope_preserved': all(hit['doc_id'] in allowed_canonical
        for hits in method_hits.values() for hit in hits),
    'search_scores_are_not_confidence': index_metadata.get('scores_are_confidence') is False,
    'full_document_route_returns_all_indexed_chunks': full_document['returned_chunks']
        == full_document['total_indexed_chunks'] and not full_document['answer_generated'],
    'followup_uses_user_queries_only': session.query_history[:2]
        == [turn['user'] for turn in followup_case['turns'][:2]],
}
for name, passed in completion_checks.items():
    print(name, ':', '통과' if passed else '확인 필요')
assert all(completion_checks.values()), '미충족 항목을 확인한 뒤 다시 실행하세요.'
print('Phase 3 범위: 청킹·검색 기준선과 연결 인터페이스')
print('생성 답변·전체 요약 품질 평가: 다음 단계')
print('OpenAI 임베딩 연결 실행:', bool(ENABLE_OPENAI and openai_build_report))
print('OpenAI dev/holdout 검색 성능 평가: 미실행')
print('공식 보고서:', ROOT / 'reports/phase3_summary.json')


source_ids_and_versions_match : 통과
canonical_content_indexed_once : 통과
chunk_and_index_content_match : 통과
source_character_audit_passed : 통과
actual_dev_query_used : 통과
selected_scope_preserved : 통과
search_scores_are_not_confidence : 통과
full_document_route_returns_all_indexed_chunks : 통과
followup_uses_user_queries_only : 통과
Phase 3 범위: 청킹·검색 기준선과 연결 인터페이스
생성 답변·전체 요약 품질 평가: 다음 단계
OpenAI 임베딩 연결 실행: False
OpenAI dev/holdout 검색 성능 평가: 미실행
공식 보고서: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase3_local_20261111/reports/phase3_summary.json
